## Oncology Plain Text

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional, Literal

# -----------------------------------------------------------------------------
# 1. Patient and Population
# -----------------------------------------------------------------------------
class PatientPopulation(BaseModel):
    cancer_type: str = Field(
        ..., description="Specific cancer type (e.g., 'Triple-Negative Breast Cancer', 'NSCLC')"
    )
    disease_stage: Optional[str] = Field(
        None, description="Stage of disease (e.g., 'Stage II-III', 'Metastatic', 'Recurrent')"
    )
    patient_age_range: Optional[str] = Field(
        None, description="Age range or median age of participants (e.g., 'Median 54 years', '18-75')"
    )
    sex_distribution: Optional[str] = Field(
        None, description="Breakdown of sex (e.g., '100% Female', '55% Male / 45% Female')"
    )
    comorbidities_summary: Optional[str] = Field(
        None, description="Summary of patient comorbidities or performance status (ECOG)"
    )
    prior_treatments_summary: Optional[str] = Field(
        None, description="Details on prior lines of therapy or treatment-naive status"
    )

# -----------------------------------------------------------------------------
# 2. Study Design
# -----------------------------------------------------------------------------
class StudyDesign(BaseModel):
    study_type: Literal['RCT', 'Prospective Cohort', 'Retrospective', 'Case Series', 'Meta-Analysis', 'Other'] = Field(
        ..., description="The methodological design of the study. 'RCT' = Randomized Controlled Trial."
    )
    sample_size: int = Field(
        ..., description="Total number of patients enrolled or analyzed in the study"
    )
    intervention: str = Field(
        ..., description="The main treatment, drug, or intervention being studied"
    )
    comparator: Optional[str] = Field(
        None, description="Control group or comparator regime (e.g., 'Placebo', 'Standard of Care'). None if single-arm."
    )
    primary_endpoint: str = Field(
        ..., description="The main outcome measure defined by the authors (e.g., 'Overall Survival', 'pCR')"
    )
    secondary_endpoints: Optional[List[str]] = Field(
        default=[], description="List of secondary outcomes measured"
    )
    followup_duration: Optional[str] = Field(
        None, description="Duration of patient follow-up (e.g., 'Median 24 months', '5 years')"
    )

# -----------------------------------------------------------------------------
# 3. Results and Reliability
# -----------------------------------------------------------------------------
class StudyResults(BaseModel):
    primary_result_text: str = Field(
        ..., description="Short textual summary of the primary outcome conclusion"
    )
    primary_result_numeric: Optional[str] = Field(
        None, description="Key numeric result if available (e.g., 'HR 0.65 (95% CI 0.5-0.8)', 'p<0.001')"
    )
    safety_profile_summary: Optional[str] = Field(
        None, description="General description of safety/tolerability and major adverse events"
    )
    limitations_summary: Optional[str] = Field(
        None, description="Author-stated limitations of the study"
    )

class OncologyStudy(BaseModel):
    filename: str = Field(..., description="Name of the source PDF file")
    title: str = Field(..., description="Extracted title of the paper")
    publication_year: Optional[int] = Field(None, description="Year of publication")

    # Nested Models
    population: PatientPopulation
    design: StudyDesign
    results: StudyResults

    # Derived Reliability Score (Calculated post-extraction, but field exists in model)
    predicted_reliability: Optional[Literal['Low', 'Medium', 'High']] = Field(
        None, description="Heuristic reliability score based on study design and sample size"
    )

## Extraction Logic

In [ ]:
import os
import json
import google.generativeai as genai
from PyPDF2 import PdfReader
from models import OncologyStudy

# Configure API Key (Best practice: set GOOGLE_API_KEY in your environment variables)
# If not set, you can hardcode it here for testing: genai.configure(api_key="YOUR_KEY")
if "GOOGLE_API_KEY" in os.environ:
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

class OncologyExtractor:
    def __init__(self, model_name="gemini-2.5-flash"):
        """
        Initialize the extractor with a specific Gemini model.
        gemini-2.0-flash is recommended for speed and cost-efficiency with long context.
        """
        self.model = genai.GenerativeModel(
            model_name=model_name,
            generation_config={"response_mime_type": "application/json"}
        )

    def extract_text_from_pdf(self, pdf_path: str, max_pages: int = 10) -> str:
        """
        Extracts text from a PDF file.
        Limits to max_pages to avoid token limits on very large appendices.
        """
        try:
            reader = PdfReader(pdf_path)
            text = ""
            # Extract text from the first N pages (usually contains Abstract, Methods, Results)
            count = min(len(reader.pages), max_pages)
            for i in range(count):
                page = reader.pages[i]
                text += page.extract_text() + "\n"
            return text
        except Exception as e:
            print(f"Error reading PDF {pdf_path}: {e}")
            return ""

    def analyze_paper(self, pdf_path: str) -> OncologyStudy:
        """
        Main method to process a PDF and return a structured OncologyStudy object.
        """
        filename = os.path.basename(pdf_path)
        print(f"Processing: {filename}...")

        # 1. Get Raw Text
        full_text = self.extract_text_from_pdf(pdf_path)
        if not full_text:
            raise ValueError(f"No text extracted from {filename}")

        # 2. Construct Prompt
        # We inject the JSON schema implicitly by asking for the specific structure
        # that matches our Pydantic model.
        prompt = f"""
        You are an expert Oncology Research Assistant. Your task is to extract structured clinical data from the following research paper text.

        Return the result primarily as a JSON object that strictly follows this structure:

        {{
            "filename": "{filename}",
            "title": "Exact title of the paper",
            "publication_year": 2024,
            "population": {{
                "cancer_type": "...",
                "disease_stage": "...",
                "patient_age_range": "...",
                "sex_distribution": "...",
                "comorbidities_summary": "...",
                "prior_treatments_summary": "..."
            }},
            "design": {{
                "study_type": "One of: RCT, Prospective Cohort, Retrospective, Case Series, Meta-Analysis, Other",
                "sample_size": 100,
                "intervention": "...",
                "comparator": "...",
                "primary_endpoint": "...",
                "secondary_endpoints": ["..."],
                "followup_duration": "..."
            }},
            "results": {{
                "primary_result_text": "...",
                "primary_result_numeric": "...",
                "safety_profile_summary": "...",
                "limitations_summary": "..."
            }}
        }}

        Ensure 'sample_size' is an integer. If a value is not found, use null.

        --- BEGIN PAPER TEXT ---
        {full_text[:30000]}
        --- END PAPER TEXT ---
        """
        # Note: We truncate text to ~30k chars (~7-8k tokens) to be safe,
        # though Gemini 1.5/2.0 can handle much more.

        # 3. Call LLM
        try:
            response = self.model.generate_content(prompt)

            # 4. Parse & Validate JSON
            # The response.text should be a JSON string thanks to response_mime_type config
            data = json.loads(response.text)

            # 5. Pydantic Validation
            study_data = OncologyStudy(**data)

            # 6. Apply Reliability Heuristic (Rule-based post-processing)
            study_data.predicted_reliability = self._calculate_reliability(study_data)

            return study_data

        except Exception as e:
            print(f"Failed to process {filename}: {e}")
            return None

    def _calculate_reliability(self, study: OncologyStudy) -> str:
        """
        Simple rule-based heuristic to assign Low/Medium/High reliability.
        """
        score = 0

        # 1. Design Weight
        if study.design.study_type == 'RCT':
            score += 3
        elif study.design.study_type == 'Prospective Cohort':
            score += 2
        elif study.design.study_type == 'Retrospective':
            score += 1

        # 2. Sample Size Weight
        if study.design.sample_size > 500:
            score += 2
        elif study.design.sample_size > 100:
            score += 1

        # 3. Comparator Weight
        if study.design.comparator and study.design.comparator.lower() != 'none':
            score += 1

        # Categorize
        if score >= 5:
            return "High"
        elif score >= 3:
            return "Medium"
        else:
            return "Low"

## Main Script

In [ ]:
import os
import json
from extractor import OncologyExtractor

# Configuration
PDF_FOLDER = "papers"  # Create this folder and put your PDFs inside
OUTPUT_FILE = "extracted_oncology_data.json"

def main():
    # 1. Setup
    if not os.path.exists(PDF_FOLDER):
        os.makedirs(PDF_FOLDER)
        print(f"Created folder '{PDF_FOLDER}'. Please add your PDF files there and run again.")
        return

    pdf_files = [f for f in os.listdir(PDF_FOLDER) if f.lower().endswith('.pdf')]

    if not pdf_files:
        print(f"No PDF files found in '{PDF_FOLDER}'.")
        return

    print(f"Found {len(pdf_files)} papers to process.")

    # 2. Initialize Extractor
    # Make sure you have set GOOGLE_API_KEY environment variable
    try:
        extractor = OncologyExtractor()
    except Exception as e:
        print("Error initializing extractor. Did you set GOOGLE_API_KEY?")
        return

    # 3. Process Batch
    results = []

    for filename in pdf_files:
        path = os.path.join(PDF_FOLDER, filename)
        try:
            study_data = extractor.analyze_paper(path)
            if study_data:
                results.append(study_data.model_dump())
                print(f"✅ Success: {filename} -> Reliability: {study_data.predicted_reliability}")
        except Exception as e:
            print(f"❌ Error processing {filename}")

    # 4. Save Results
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2)

    print(f"\nExtraction complete. Data saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

## Download PDFs

In [ ]:
import os
import requests
import time

# Create papers directory if it doesn't exist
OUTPUT_DIR = "papers"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Dictionary of papers: "Filename.pdf": "URL"
# I have selected direct PDF links where possible to ensure successful download.
PAPERS = {
    # --- HIGH RELIABILITY (RCTs & Large Trials) ---
    "KEYNOTE-522_FDA.pdf": "https://www.accessdata.fda.gov/drugsatfda_docs/nda/2021/761034Orig1s029MedR.pdf",
    "PALOMA-2_FDA.pdf": "https://www.accessdata.fda.gov/drugsatfda_docs/nda/2015/207103Orig1s000StatR.pdf",
    "MONALEESA-2_FDA.pdf": "https://www.accessdata.fda.gov/drugsatfda_docs/nda/2017/209092Orig1s000MedR.pdf", # Subst. for ClinicalTrials.gov link
    "DREAMseq_Results.pdf": "https://ecog-acrin.org/wp-content/uploads/2024/02/EA6134_ClinTrialResultsSumm_EA_Web_up08Jan2024.pdf",
    "E1910_NICE.pdf": "https://www.nice.org.uk/guidance/ta1049/documents/1",
    "PATINA_Pfizer.pdf": "https://www.pfizer.com/print/pdf/node/561212",
    "Deferred_Nephrectomy_BMC.pdf": "https://bmccancer.biomedcentral.com/counter/pdf/10.1186/s12885-024-11987-3.pdf",
    "Personalized_Oncology_Meta_PLOS.pdf": "https://journals.plos.org/plosone/article/file?id=10.1371/journal.pone.0332599&type=printable",
    
    # --- MEDIUM RELIABILITY (Real-World & Observational) ---
    "RealWorld_Palbociclib_JCO.pdf": "https://ascopubs.org/doi/pdf/10.1200/JGO.18.00239",
    "Lenvatinib_RealWorld_PMC.pdf": "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8573180/pdf/fonc-11-729904.pdf",
    "Pancreatic_RealWorld_Lee_PMC.pdf": "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9856436/pdf/cancers-15-00438.pdf",
    "Cervical_Cancer_Elderly_JAMA.pdf": "https://jamanetwork.com/journals/jamanetworkopen/articlepdf/2822206/jamanetworkopen_lin_2024_oi_240898_1723134691.68886.pdf",
    "Precision_Medicine_Feasibility_PLOS.pdf": "https://journals.plos.org/plosone/article/file?id=10.1371/journal.pone.0325769&type=printable",
    
    # --- LOW RELIABILITY (Retrospective & Small Sample) ---
    "Merkel_Cell_Retrospective_PMC.pdf": "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6210570/pdf/curr-oncol-25-e451.pdf",
    "Glioblastoma_Retrospective_PMC.pdf": "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC11180423/pdf/cureus-0016-00000060424.pdf",
    "SingleArm_PhaseII_Design_PMC.pdf": "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5223772/pdf/nihms823772.pdf",
    "Oral_Fenbendazole_Review.pdf": "https://ar.iiarjournals.org/content/anticanres/44/9/3725.full.pdf"
}

# Headers to mimic a real browser (prevents 403 Forbidden errors on some sites)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

def download_file(url, filename):
    filepath = os.path.join(OUTPUT_DIR, filename)
    print(f"⬇️  Downloading: {filename}...")
    
    try:
        response = requests.get(url, headers=HEADERS, stream=True, timeout=30)
        
        # Check if the request was successful
        if response.status_code == 200:
            # Check content type if possible
            content_type = response.headers.get('Content-Type', '').lower()
            if 'html' in content_type and 'pdf' not in url:
                print(f"⚠️  Warning: {filename} might be an HTML page, not a PDF. (Content-Type: {content_type})")
            
            with open(filepath, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"✅ Saved to {filepath}")
        else:
            print(f"❌ Failed: HTTP {response.status_code}")
            
    except Exception as e:
        print(f"❌ Error: {e}")

def main():
    print(f"Starting download of {len(PAPERS)} papers into '/{OUTPUT_DIR}'...\n")
    
    for filename, url in PAPERS.items():
        download_file(url, filename)
        # Be polite to servers
        time.sleep(1) 
        
    print("\n🎉 Download complete! Run 'main.py' to start extracting.")

if __name__ == "__main__":
    main()

main()

In [ ]:
!ls